In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')
        
def fill_with_mode(df, feature_list):
    for feature in feature_list:
        mode = df[feature].mode()[0]
        df[feature].fillna(value=mode, inplace=True)

def fill_with_mean(df, feature_list):
    for feature in feature_list:
        df[feature] = pd.to_numeric(df[feature], errors='coerce')
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)

In [3]:
df_dk = pd.read_csv("Data/y/Denmark.csv")

In [4]:
df_dk = df_dk.loc[6:20]

In [5]:
df_dk = df_dk.copy()
df_dk['LFG'] = df_dk['LF'].diff()
df_dk['EPFRG'] = df_dk['EPFR'].diff()
df_dk['GGAG'] = df_dk['GGA'].diff()
df_dk['PSG'] = df_dk['PS'].diff()
df_dk = df_dk.drop(6)

In [6]:
df_dk = df_dk.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [7]:
df_dk["EMP"] = df_dk["EMP"].replace(':', np.nan)
df_dk["LTU"] = df_dk["LTU"].replace(':', np.nan)
df_dk["FEM"] = df_dk["FEM"].replace(':', np.nan)

In [8]:
missing_values(df_dk)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [9]:
fill_with_mode(df_dk, ["QOR", "QOA"])

In [10]:
fill_with_mean(df_dk, ["EMP", "LTU", "FEM"])

In [11]:
dk_context_2007_2013 = pd.read_csv("Data/Context/Denmark_2007_2013.csv")

In [12]:
dk_context_2007_2013 = dk_context_2007_2013[["TIME_PERIOD", "DI", "GE", "CPI", "RQ", "VAA"]]

In [13]:
eu_payments = pd.read_csv("Raw_Data/Variables/Historic_EU_payments_annual_timeseries.csv")

In [14]:
eu_payments = eu_payments.drop(['NUTS1_ID', 'NUTS2 code+name', 'NUTS2_ID', 'NUTS2_name', 'Modelled_annual_expenditure',
                  'Standard_Deviation_of_annual_expenditure', 'Standard_Error_of_modelled_annual_expenditure'], axis=1)

In [15]:
dk_eu_payments = eu_payments[eu_payments['Country'] == 'DK'].copy()

In [16]:
dk_eu_payments

,Country,Fund,Year,Programming_Period,EU_Payment_annual
833,DK,ERDF,2007,2007-2013,"1,194,717"
834,DK,ERDF,2008,2007-2013,"1,792,076"
835,DK,ERDF,2009,2007-2013,"2,228,417"
836,DK,ERDF,2010,2007-2013,"4,678,038"
837,DK,ERDF,2011,2007-2013,"15,425,586"
...,...,...,...,...,...
2311,DK,ESF,2016,2014-2020,"67,409"
2312,DK,ESF,2017,2014-2020,"1,443,642"
2313,DK,ESF,2018,2014-2020,"2,077,670"
2314,DK,ESF,2019,2014-2020,"2,615,692"


In [17]:
dk_eu_payments['EU_Payment_annual'] = pd.to_numeric(dk_eu_payments['EU_Payment_annual'].str.replace(',', ''))

In [18]:
dk_eu_payments = dk_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

In [19]:
dk_eu_payments = dk_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [20]:
dk_eu_payments_2007_2013 = dk_eu_payments.loc[0:6]

In [21]:
df_dk_2007_2013 = df_dk.loc[7:13]

In [22]:
df_dk_2007_2013 = pd.merge(df_dk_2007_2013, dk_context_2007_2013, on='TIME_PERIOD', how='outer')

In [23]:
df_dk_2007_2013 = pd.merge(df_dk_2007_2013, dk_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

In [24]:
df_dk_2007_2013['EUPC'] = df_dk_2007_2013['EU_Payment_annual'] / df_dk_2007_2013['PS']

In [25]:
df_dk_2007_2013 = df_dk_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [26]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_dk_2007_2013.values)

df_dk_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_dk_2007_2013.index, columns=df_dk_2007_2013.columns)

In [27]:
df_dk_2007_2013_scaled.to_csv('Data/y/dk_2007_2013_tcdf.csv')

In [28]:
dk_context_2014_2020 = pd.read_csv("Data/Context/Denmark_2014_2020.csv")

dk_context_2014_2020 = dk_context_2014_2020[["TIME_PERIOD", "DI", "COC", "CPI", "ROL", "VAA"]]

df_dk_2014_2020 = df_dk.loc[14:20]

dk_eu_payments_2014_2020 = dk_eu_payments.loc[7:13]

df_dk_2014_2020 = pd.merge(df_dk_2014_2020, dk_context_2014_2020, on='TIME_PERIOD', how='outer')

df_dk_2014_2020 = pd.merge(df_dk_2014_2020, dk_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_dk_2014_2020['EUPC'] = df_dk_2014_2020['EU_Payment_annual'] / df_dk_2014_2020['PS']

df_dk_2014_2020 = df_dk_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [29]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_dk_2014_2020.values)

df_dk_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_dk_2014_2020.index, columns=df_dk_2014_2020.columns)

In [30]:
df_dk_2014_2020_scaled.to_csv('Data/y/dk_2014_2020_tcdf.csv')

====================================================================

In [31]:
df_de = pd.read_csv("Data/y/Germany.csv")

In [32]:
df_de = df_de.loc[6:20]

In [33]:
df_de = df_de.copy()
df_de['LFG'] = df_de['LF'].diff()
df_de['EPFRG'] = df_de['EPFR'].diff()
df_de['GGAG'] = df_de['GGA'].diff()
df_de['PSG'] = df_de['PS'].diff()
df_de = df_de.drop(6)

In [34]:
df_de = df_de.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [35]:
df_de["EMP"] = df_de["EMP"].replace(':', np.nan)
df_de["LTU"] = df_de["LTU"].replace(':', np.nan)
df_de["FEM"] = df_de["FEM"].replace(':', np.nan)

In [36]:
missing_values(df_de)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [37]:
df_de = df_de.copy()
fill_with_mode(df_de, ["QOR"])

In [38]:
fill_with_mean(df_de, ["EMP", "LTU", "FEM", "QOA"])

In [39]:
de_context_2007_2013 = pd.read_csv("Data/Context/Germany_2007_2013.csv")

In [40]:
de_context_2007_2013 = de_context_2007_2013[["TIME_PERIOD", "PMSI", "JI", "DBP", "DWCP", "PR"]]

In [41]:
de_eu_payments = eu_payments[eu_payments['Country'] == 'DE'].copy()

In [42]:
de_eu_payments

,Country,Fund,Year,Programming_Period,EU_Payment_annual
0,DE,ERDF,2007,2007-2013,"1,017,013"
1,DE,ERDF,2008,2007-2013,"1,525,519"
2,DE,ERDF,2009,2007-2013,"1,271,266"
3,DE,ERDF,2010,2007-2013,"3,453,592"
4,DE,ERDF,2011,2007-2013,"2,385,830"
...,...,...,...,...,...
2251,DE,ESF,2016,2014-2020,"1,162,269"
2252,DE,ESF,2017,2014-2020,"95,111,127"
2253,DE,ESF,2018,2014-2020,"52,908,298"
2254,DE,ESF,2019,2014-2020,"149,834,127"


In [43]:
de_eu_payments['EU_Payment_annual'] = pd.to_numeric(de_eu_payments['EU_Payment_annual'].str.replace(',', ''))

In [44]:
de_eu_payments = de_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

In [45]:
de_eu_payments = de_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [46]:
de_eu_payments_2007_2013 = de_eu_payments.loc[0:6]

In [47]:
df_de_2007_2013 = df_de.loc[7:13]

In [48]:
df_de_2007_2013 = pd.merge(df_de_2007_2013, de_context_2007_2013, on='TIME_PERIOD', how='outer')

In [49]:
df_de_2007_2013 = pd.merge(df_de_2007_2013, de_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

In [50]:
df_de_2007_2013['EUPC'] = df_de_2007_2013['EU_Payment_annual'] / df_de_2007_2013['PS']

In [51]:
df_de_2007_2013 = df_de_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [52]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_de_2007_2013.values)

df_de_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_de_2007_2013.index, columns=df_de_2007_2013.columns)

In [53]:
df_de_2007_2013_scaled.to_csv('Data/y/de_2007_2013_tcdf.csv')

In [54]:
df_de.columns.equals(df_dk.columns)

True

In [55]:
de_context_2014_2020 = pd.read_csv("Data/Context/Germany_2014_2020.csv")

de_context_2014_2020 = de_context_2014_2020[["TIME_PERIOD", "DI", "COC", "CPI", "ROL", "VAA"]]

df_de_2014_2020 = df_de.loc[14:20]

de_eu_payments_2014_2020 = de_eu_payments.loc[7:13]

df_de_2014_2020 = pd.merge(df_de_2014_2020, de_context_2014_2020, on='TIME_PERIOD', how='outer')

df_de_2014_2020 = pd.merge(df_de_2014_2020, de_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_de_2014_2020['EUPC'] = df_de_2014_2020['EU_Payment_annual'] / df_de_2014_2020['PS']

df_de_2014_2020 = df_de_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [56]:
df_de_2014_2020

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,LFG,EPFRG,GGAG,PSG,DI,COC,CPI,ROL,VAA,EUPC
0,4.98,4.63,7.61,0.44,5.88,5.940000,1.43,232.0,20.96,13.18,...,0.27,13598.00,-40820.0,0.33,8.64,1.81,79.0,1.85,1.44,3.929514e+07
1,4.61,4.20,7.33,1.82,5.72,5.950000,1.39,234.0,21.05,13.21,...,0.24,26869.00,4675.0,0.71,8.64,1.78,81.0,1.76,1.42,2.798182e+07
2,4.10,3.72,7.08,1.64,5.60,5.900000,1.31,236.0,21.13,13.33,...,0.43,-682.00,3007.0,0.66,8.63,1.78,81.0,1.58,1.36,8.995473e+06
3,3.78,3.32,6.77,2.89,5.50,5.800000,1.28,237.0,21.29,13.46,...,0.27,27047.00,-17289.0,0.31,8.61,1.80,81.0,1.57,1.43,2.219124e+07
4,3.38,2.90,6.22,4.00,5.50,5.600000,1.25,237.0,21.45,13.56,...,0.22,8209.00,-28666.0,0.25,8.68,1.90,80.0,1.59,1.44,2.595348e+07
5,3.16,2.72,5.86,1.91,5.30,5.500000,1.20,238.0,21.63,13.66,...,0.37,16219.01,-50838.0,0.18,8.68,1.87,80.0,1.58,1.35,2.834817e+07
6,3.88,3.49,7.86,4.49,5.50,6.166923,1.22,238.0,21.84,13.76,...,-0.61,12158.99,-58901.0,0.07,8.67,1.83,80.0,1.52,1.37,2.796353e+07


In [57]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_de_2014_2020.values)

df_de_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_de_2014_2020.index, columns=df_de_2014_2020.columns)

In [58]:
df_de_2014_2020_scaled.to_csv('Data/y/de_2014_2020_tcdf.csv')

===================================================================================================================

In [59]:
df_pl = pd.read_csv("Data/y/Poland.csv")

In [60]:
df_pl = df_pl.loc[6:20]

In [61]:
df_pl

,Unnamed: 0,REF_AREA_LABEL,TIME_PERIOD,HI,LF,UR,URFF,YUA,FDI,QOR,...,EIA,EBSG,GDPC,GDP,EIS,NINA,LBGD,EMP,LTU,FEM
6,6,Poland,2006,NaN,17.28,13.79,14.83,29.54,6.21,3.20,...,15.789906,-2.384721,28142.090823,1.313753e+12,54.230174,1.838500,-3.262140,:,:,NaN
7,7,Poland,2007,NaN,17.27,9.55,10.23,21.39,5.83,2.57,...,14.744292,-3.803166,31201.797665,1.402569e+12,54.531280,2.446382,-1.872662,:,:,:
8,8,Poland,2008,NaN,17.50,7.07,7.85,17.10,2.72,1.91,...,13.971372,-5.237136,33846.381917,1.464054e+12,54.138756,2.544367,-3.397579,:,:,:
9,9,Poland,2009,NaN,17.81,8.13,8.55,20.47,3.18,2.06,...,13.284544,-0.960496,36057.305377,1.502341e+12,55.604948,2.384192,-6.157262,62.6,2.6,55.6
10,10,Poland,2010,NaN,18.03,9.58,9.87,23.58,3.94,2.23,...,13.054421,-2.441439,37895.481599,1.549949e+12,56.642149,2.757119,-6.142614,62.2,3.1,55.6
11,11,Poland,2011,NaN,18.10,9.58,10.24,25.53,3.57,2.33,...,12.908103,-2.287307,41087.789260,1.631406e+12,56.418280,3.145783,-4.112383,62.5,3.7,55.8
12,12,Poland,2012,NaN,18.23,10.03,10.77,26.31,1.53,2.62,...,12.578914,-0.784100,42619.735974,1.656070e+12,57.005312,2.489764,-3.437896,62.9,4.2,56.2
13,13,Poland,2013,5.82,18.27,10.29,11.01,27.16,0.26,3.05,...,11.998832,1.117069,43053.616233,1.667401e+12,57.460898,1.935317,-4.023830,63.2,4.5,56.3
14,14,Poland,2014,NaN,18.39,8.97,9.52,23.73,3.85,3.55,...,11.490861,0.407324,44990.974498,1.732772e+12,57.966922,2.013741,-3.367862,64.9,3.9,58.2
15,15,Poland,2015,5.79,18.34,7.47,7.64,20.68,3.30,3.81,...,11.534171,2.067846,47637.139301,1.809564e+12,57.928061,2.243049,-2.532527,66.3,3.0,59.9


In [62]:
df_pl = df_pl.copy()
df_pl['LFG'] = df_pl['LF'].diff()
df_pl['EPFRG'] = df_pl['EPFR'].diff()
df_pl['GGAG'] = df_pl['GGA'].diff()
df_pl['PSG'] = df_pl['PS'].diff()
df_pl = df_pl.drop(6)

In [63]:
df_pl = df_pl.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [64]:
df_pl["EMP"] = df_pl["EMP"].replace(':', np.nan)
df_pl["LTU"] = df_pl["LTU"].replace(':', np.nan)
df_pl["FEM"] = df_pl["FEM"].replace(':', np.nan)

In [65]:
missing_values(df_pl)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [66]:
df_pl = df_pl.copy()
fill_with_mode(df_pl, ["QOR", "QOA", "FEM"])

In [67]:
fill_with_mean(df_pl, ["EMP", "LTU"])

In [68]:
pl_context_2007_2013 = pd.read_csv("Data/Context/Poland_2007_2013.csv")

In [69]:
pl_context_2007_2013 = pl_context_2007_2013[["TIME_PERIOD", "PR", "RQ", "COC", "ROL", "CPI"]]

In [70]:
pl_eu_payments = eu_payments[eu_payments['Country'] == 'PL'].copy()

In [71]:
pl_eu_payments

,Country,Fund,Year,Programming_Period,EU_Payment_annual
1125,PL,CF,2007,2007-2013,"45,219,844"
1126,PL,CF,2008,2007-2013,"72,351,752"
1127,PL,CF,2009,2007-2013,"129,326,232"
1128,PL,CF,2010,2007-2013,"108,332,216"
1129,PL,CF,2011,2007-2013,"207,533,488"
...,...,...,...,...,...
2716,PL,ESF,2016,2014-2020,"21,309,721"
2717,PL,ESF,2017,2014-2020,"64,680,089"
2718,PL,ESF,2018,2014-2020,"110,564,312"
2719,PL,ESF,2019,2014-2020,"98,266,373"


In [72]:
pl_eu_payments['EU_Payment_annual'] = pd.to_numeric(pl_eu_payments['EU_Payment_annual'].str.replace(',', ''))

In [73]:
pl_eu_payments = pl_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

In [74]:
pl_eu_payments = pl_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [75]:
pl_eu_payments_2007_2013 = pl_eu_payments.loc[0:6]

In [76]:
df_pl_2007_2013 = df_pl.loc[7:13]

In [77]:
df_pl_2007_2013 = pd.merge(df_pl_2007_2013, pl_context_2007_2013, on='TIME_PERIOD', how='outer')

In [78]:
df_pl_2007_2013 = pd.merge(df_pl_2007_2013, pl_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

In [79]:
df_pl_2007_2013['EUPC'] = df_pl_2007_2013['EU_Payment_annual'] / df_pl_2007_2013['PS']

In [80]:
df_pl_2007_2013 = df_pl_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [81]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_pl_2007_2013.values)

df_pl_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_pl_2007_2013.index, columns=df_pl_2007_2013.columns)

In [82]:
df_pl_2007_2013_scaled.to_csv('Data/y/pl_2007_2013_tcdf.csv')

In [83]:
df_de.columns.equals(df_pl.columns)

True

In [84]:
pl_context_2014_2020 = pd.read_csv("Data/Context/Poland_2014_2020.csv")

pl_context_2014_2020 = pl_context_2014_2020[["TIME_PERIOD", "JI", "FOP", "DBD", "PT", "PR"]]

df_pl_2014_2020 = df_pl.loc[14:20]

pl_eu_payments_2014_2020 = pl_eu_payments.loc[7:13]

df_pl_2014_2020 = pd.merge(df_pl_2014_2020, pl_context_2014_2020, on='TIME_PERIOD', how='outer')

df_pl_2014_2020 = pd.merge(df_pl_2014_2020, pl_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_pl_2014_2020['EUPC'] = df_pl_2014_2020['EU_Payment_annual'] / df_pl_2014_2020['PS']

df_pl_2014_2020 = df_pl_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [85]:
df_pl_2014_2020

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,LFG,EPFRG,GGAG,PSG,JI,FOP,DBD,PT,PR,EUPC
0,8.97,9.52,23.73,3.85,3.55,4.02,11.49,124.0,14.88,15.09,...,0.12,3032.62,-13172.0,-0.03,0.957,0.975,37.0,72.292209,60,3.074712e+08
1,7.47,7.64,20.68,3.30,3.81,4.07,11.53,124.0,15.38,15.10,...,-0.05,3229.87,3157.0,-0.02,0.874,0.965,37.0,72.258398,60,1.699951e+08
2,6.14,6.18,17.59,3.82,4.00,4.30,10.58,124.0,15.91,15.13,...,0.05,-285.22,10799.0,-0.02,0.783,0.745,37.0,79.141239,65,7.626680e+07
3,4.87,4.86,14.76,2.38,4.10,4.50,10.22,124.0,16.47,15.21,...,0.07,904.51,13375.0,0.00,0.690,0.709,37.0,79.105836,61,1.733203e+08
4,3.84,3.82,11.68,3.35,4.10,4.80,9.62,124.0,17.01,15.31,...,-0.03,-1915.52,-1790.0,0.00,0.595,0.709,37.0,79.495816,62,2.872052e+08
5,3.27,3.57,9.83,3.15,4.30,4.80,9.15,124.0,17.56,15.42,...,-0.05,3853.76,-17545.0,0.00,0.572,0.704,37.0,76.490761,62,2.826689e+08
6,3.15,3.28,10.83,3.31,4.10,3.65,9.59,124.0,18.08,15.49,...,-0.25,2608.05,-16997.0,-0.45,0.570,0.707,37.0,76.831746,63,3.222559e+08


In [86]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_pl_2014_2020.values)

df_pl_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_pl_2014_2020.index, columns=df_pl_2014_2020.columns)

In [87]:
df_pl_2014_2020_scaled.to_csv('Data/y/pl_2014_2020_tcdf.csv')

==================================================================================================================

In [88]:
df_lt = pd.read_csv("Data/y/Lithuania.csv")

In [89]:
df_lt = df_lt.loc[6:20]

In [90]:
df_lt

,Unnamed: 0,REF_AREA_LABEL,TIME_PERIOD,HI,LF,UR,URFF,YUA,FDI,QOR,...,EIA,EBSG,GDPC,GDP,EIS,NINA,LBGD,EMP,LTU,FEM
6,6,Lithuania,2006,NaN,1.51,5.78,5.61,9.98,7.50,4.92,...,13.832556,-10.283491,7339.529632,3.546242e+10,56.596662,3.118363,0.075001,:,:,NaN
7,7,Lithuania,2007,NaN,1.51,4.25,4.33,8.39,6.55,4.97,...,11.325410,-13.094653,8985.274630,3.939093e+10,58.126069,3.723648,-0.493558,:,:,:
8,8,Lithuania,2008,NaN,1.51,5.83,5.63,13.35,3.61,5.21,...,8.064581,-11.598256,10219.116755,4.041488e+10,61.418727,3.480952,-2.844590,:,:,:
9,9,Lithuania,2009,NaN,1.52,13.78,10.52,29.58,-0.96,5.28,...,8.983906,-1.755188,8528.034573,3.441787e+10,64.167134,2.773311,-8.626258,67.0,3.3,67.2
10,10,Lithuania,2010,NaN,1.49,17.81,14.52,35.74,3.01,5.27,...,8.830495,-2.029481,8931.519313,3.456490e+10,66.610059,2.591630,-7.016844,64.3,7.4,65.0
11,11,Lithuania,2011,NaN,1.48,15.39,12.91,32.59,4.36,5.23,...,8.486140,-2.706873,10259.331961,3.675066e+10,66.955836,2.373246,-5.541350,66.9,8.0,66.6
12,12,Lithuania,2012,NaN,1.47,13.36,11.57,26.68,1.59,5.17,...,8.796753,0.613211,11120.403233,3.836141e+10,66.119954,1.956453,-2.893765,68.5,6.6,67.9
13,13,Lithuania,2013,5.43,1.47,11.77,10.48,21.90,1.66,5.02,...,8.422800,1.457239,11776.792571,3.991386e+10,66.052053,2.151628,-2.417272,69.9,5.1,68.6
14,14,Lithuania,2014,NaN,1.48,10.70,9.22,19.26,0.74,4.94,...,9.165997,1.930309,12395.979849,4.141963e+10,66.110306,2.113033,-1.911556,71.8,4.8,70.6
15,15,Lithuania,2015,5.83,1.47,9.12,8.16,16.30,2.50,4.96,...,9.069029,-0.986813,12861.323296,4.259297e+10,65.858411,2.545603,-1.093729,73.3,3.9,72.2


In [91]:
df_lt = df_lt.copy()
df_lt['LFG'] = df_lt['LF'].diff()
df_lt['EPFRG'] = df_lt['EPFR'].diff()
df_lt['GGAG'] = df_lt['GGA'].diff()
df_lt['PSG'] = df_lt['PS'].diff()
df_lt = df_lt.drop(6)

In [92]:
df_lt = df_lt.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [93]:
df_lt["EMP"] = df_lt["EMP"].replace(':', np.nan)
df_lt["LTU"] = df_lt["LTU"].replace(':', np.nan)
df_lt["FEM"] = df_lt["FEM"].replace(':', np.nan)

In [94]:
missing_values(df_lt)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [95]:
df_lt = df_lt.copy()
fill_with_mode(df_lt, ["QOR", "QOA"])

In [96]:
fill_with_mean(df_lt, ["EMP", "LTU", "FEM"])

In [97]:
lt_context_2007_2013 = pd.read_csv("Data/Context/Lithuania_2007_2013.csv")

In [98]:
lt_context_2007_2013 = lt_context_2007_2013[["TIME_PERIOD", "PR", "RQ", "COC", "ROL", "CPI"]]

In [99]:
lt_eu_payments = eu_payments[eu_payments['Country'] == 'LT'].copy()

In [100]:
lt_eu_payments['EU_Payment_annual'] = pd.to_numeric(lt_eu_payments['EU_Payment_annual'].str.replace(',', ''))

In [101]:
lt_eu_payments = lt_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

In [102]:
lt_eu_payments = lt_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [103]:
lt_eu_payments_2007_2013 = lt_eu_payments.loc[0:6]

In [104]:
df_lt_2007_2013 = df_lt.loc[7:13]

In [105]:
df_lt_2007_2013 = pd.merge(df_lt_2007_2013, lt_context_2007_2013, on='TIME_PERIOD', how='outer')

In [106]:
df_lt_2007_2013 = pd.merge(df_lt_2007_2013, lt_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

In [107]:
df_lt_2007_2013['EUPC'] = df_lt_2007_2013['EU_Payment_annual'] / df_lt_2007_2013['PS']

In [108]:
df_lt_2007_2013 = df_lt_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [109]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_lt_2007_2013.values)

df_lt_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_lt_2007_2013.index, columns=df_lt_2007_2013.columns)

In [110]:
df_lt_2007_2013_scaled.to_csv('Data/y/lt_2007_2013_tcdf.csv')

In [111]:
df_pl.columns.equals(df_lt.columns)

True

In [112]:
lt_context_2014_2020 = pd.read_csv("Data/Context/Lithuania_2014_2020.csv")

lt_context_2014_2020 = lt_context_2014_2020[["TIME_PERIOD", "EC", "DI", "SSB", "COC", "CPI"]]

df_lt_2014_2020 = df_lt.loc[14:20]

lt_eu_payments_2014_2020 = lt_eu_payments.loc[7:13]

df_lt_2014_2020 = pd.merge(df_lt_2014_2020, lt_context_2014_2020, on='TIME_PERIOD', how='outer')

df_lt_2014_2020 = pd.merge(df_lt_2014_2020, lt_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_lt_2014_2020['EUPC'] = df_lt_2014_2020['EU_Payment_annual'] / df_lt_2014_2020['PS']

df_lt_2014_2020 = df_lt_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [113]:
df_lt_2014_2020

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,LFG,EPFRG,GGAG,PSG,EC,DI,SSB,COC,CPI,EUPC
0,10.70,9.22,19.26,0.74,4.94,4.19,9.17,47.0,18.60,14.58,...,0.01,107.00,-452.0,-0.02,78.802445,7.54,92.975890,0.55,58.0,3.446086e+08
1,9.12,8.16,16.30,2.50,4.96,4.12,9.07,46.0,18.88,14.61,...,-0.01,217.00,260.0,-0.03,78.802445,7.54,92.436369,0.58,61.0,2.963516e+07
2,7.86,6.65,14.45,2.74,4.90,4.40,7.98,46.0,19.18,14.74,...,0.01,312.00,25.0,-0.03,78.802445,7.47,92.931332,0.68,59.0,1.155739e+08
3,7.07,5.67,13.27,2.90,4.70,4.40,7.78,45.0,19.50,14.91,...,-0.01,294.85,-79.0,-0.04,78.802445,7.41,92.984744,0.52,59.0,1.708537e+08
4,6.15,5.41,11.12,2.40,4.70,4.60,7.18,45.0,19.73,15.05,...,0.00,-161.65,86.0,-0.02,78.802445,7.50,93.053500,0.47,59.0,3.889764e+08
5,6.26,5.49,11.85,6.23,4.80,4.90,6.44,45.0,19.86,15.11,...,0.01,350.20,128.0,-0.01,78.802445,7.50,93.175686,0.67,60.0,1.883568e+08
6,8.49,7.69,19.56,7.94,4.70,4.40,5.69,45.0,19.93,15.12,...,0.01,150.30,5.0,0.00,78.802445,7.13,93.273709,0.78,60.0,5.145216e+08


In [114]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_lt_2014_2020.values)

df_lt_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_lt_2014_2020.index, columns=df_lt_2014_2020.columns)

In [115]:
df_lt_2014_2020_scaled.to_csv('Data/y/lt_2014_2020_tcdf.csv')

===================================================================================================================

In [116]:
df_lv = pd.read_csv("Data/y/Latvia.csv")

In [117]:
df_lv = df_lv.loc[6:20]

In [118]:
df_lv

,Unnamed: 0,REF_AREA_LABEL,TIME_PERIOD,HI,LF,UR,URFF,YUA,FDI,QOR,...,EIA,EBSG,GDPC,GDP,EIS,NINA,LBGD,EMP,LTU,FEM
6,6,Latvia,2006,NaN,1.12,7.03,6.74,13.61,8.35,3.44,...,11.460996,-21.789187,7345.439440,2.634463e+10,61.169864,3.919721,-0.039049,:,:,NaN
7,7,Latvia,2007,NaN,1.14,6.05,5.57,10.56,9.22,3.11,...,10.179928,-20.108026,9775.522707,2.908830e+10,61.186668,3.950384,0.134663,:,:,:
8,8,Latvia,2008,NaN,1.15,7.74,7.07,13.65,4.20,2.85,...,7.957091,-13.299099,10725.051692,2.810258e+10,62.643879,2.458268,-3.172711,:,:,:
9,9,Latvia,2009,NaN,1.10,17.51,14.13,33.43,-0.58,3.19,...,8.750842,-2.039931,8629.945617,2.359492e+10,66.906110,1.513146,-7.966839,66.3,5.0,66.2
10,10,Latvia,2010,NaN,1.06,19.48,16.29,36.24,2.02,3.11,...,8.614454,-2.131443,8447.759415,2.273105e+10,68.311167,1.546866,-8.375610,63.9,9.7,64.2
11,11,Latvia,2011,NaN,1.03,16.21,13.79,31.07,5.72,3.10,...,8.895122,-6.131271,9281.534430,2.342124e+10,68.196401,2.592385,-3.827185,66.1,9.8,65.1
12,12,Latvia,2012,NaN,1.04,15.05,13.94,28.56,3.99,3.16,...,8.373286,-5.548474,10374.147811,2.513222e+10,68.149704,2.608442,-1.006694,67.9,8.6,66.2
13,13,Latvia,2013,5.05,1.02,11.87,11.14,23.24,3.39,3.00,...,8.056219,-4.525224,10909.112726,2.565901e+10,68.022089,2.306056,-0.749488,69.5,6.3,67.5
14,14,Latvia,2014,NaN,1.01,10.85,9.83,19.59,3.45,3.09,...,7.503669,-2.898049,11430.790829,2.619656e+10,68.648829,2.458454,-1.357478,70.6,5.1,68.4
15,15,Latvia,2015,5.10,1.01,9.87,8.64,16.32,3.09,3.31,...,7.939155,-1.188889,12007.048703,2.718767e+10,68.415218,3.558658,-1.740458,72.5,4.9,70.5


In [119]:
df_lv = df_lv.copy()
df_lv['LFG'] = df_lv['LF'].diff()
df_lv['EPFRG'] = df_lv['EPFR'].diff()
df_lv['GGAG'] = df_lv['GGA'].diff()
df_lv['PSG'] = df_lv['PS'].diff()
df_lv = df_lv.drop(6)

In [120]:
df_lv = df_lv.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [121]:
df_lv["EMP"] = df_lv["EMP"].replace(':', np.nan)
df_lv["LTU"] = df_lv["LTU"].replace(':', np.nan)
df_lv["FEM"] = df_lv["FEM"].replace(':', np.nan)

In [122]:
missing_values(df_lv)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [123]:
# df_lv = df_lv.copy()
# fill_with_mode(df_lv, ["QOR", "QOA"])

In [124]:
fill_with_mean(df_lv, ["EMP", "LTU", "FEM", "QOR", "QOA"])

In [125]:
lv_context_2007_2013 = pd.read_csv("Data/Context/Latvia_2007_2013.csv")

In [126]:
lv_context_2007_2013 = lv_context_2007_2013[["TIME_PERIOD", "PR", "RQ", "COC", "ROL", "CPI"]]

In [127]:
lv_eu_payments = eu_payments[eu_payments['Country'] == 'LV'].copy()

lv_eu_payments['EU_Payment_annual'] = pd.to_numeric(lv_eu_payments['EU_Payment_annual'].str.replace(',', ''))

lv_eu_payments = lv_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

lv_eu_payments = lv_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [128]:
lv_eu_payments_2007_2013 = lv_eu_payments.loc[0:6]

In [129]:
df_lv_2007_2013 = df_lv.loc[7:13]

df_lv_2007_2013 = pd.merge(df_lv_2007_2013, lv_context_2007_2013, on='TIME_PERIOD', how='outer')

df_lv_2007_2013 = pd.merge(df_lv_2007_2013, lv_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

df_lv_2007_2013['EUPC'] = df_lv_2007_2013['EU_Payment_annual'] / df_lv_2007_2013['PS']

df_lv_2007_2013 = df_lv_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [130]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_lv_2007_2013.values)

df_lv_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_lv_2007_2013.index, columns=df_lv_2007_2013.columns)

In [131]:
df_lv_2007_2013_scaled

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,LFG,EPFRG,GGAG,PSG,PR,RQ,COC,ROL,CPI,EUPC
0,-1.576146,-1.659162,-1.604288,1.855612,0.334555,-6.978189e-15,2.167992,1.248075,-1.489977,-0.722574,...,1.341641,-0.760100,0.801231,1.080123e+00,-0.866025,-0.425864,1.567686,-1.805461,0.232845,-1.522454
1,-1.214513,-1.253452,-1.266831,0.073048,-2.101004,1.571348e+00,-1.062170,1.248075,-1.050164,-1.139116,...,0.950329,-0.749043,-0.442589,1.080123e+00,1.154701,0.070977,-0.550065,0.809345,0.776151,-1.289585
2,0.876112,0.656092,0.893334,-1.624295,1.083958,7.071068e-01,0.087302,0.277350,-0.505635,-0.663068,...,-1.397542,-0.877788,-1.164070,-1.080123e+00,1.154701,-1.171126,-1.127634,1.245146,-0.582113,-0.072913
3,1.297661,1.240315,1.200213,-0.701054,0.334555,3.928371e-01,-0.116402,0.277350,0.059838,-0.306031,...,-1.006231,-0.655221,1.969965,-1.080123e+00,1.154701,-0.922705,-1.127634,-0.062257,-1.125418,0.137642
4,0.597933,0.564131,0.635600,0.612788,0.240879,-1.728483e+00,0.305556,-0.693375,0.541538,0.051005,...,-0.614919,-0.010291,-0.937179,-1.080123e+00,-0.866025,-0.425864,0.412549,-0.933859,-1.397071,0.397817
5,0.349711,0.604702,0.361484,-0.001522,0.802932,-9.428090e-01,-0.465609,-0.693375,0.960407,0.765078,...,0.950329,1.490532,-0.218963,-2.061087e-14,-0.866025,1.064660,-0.357542,0.809345,0.504498,1.370355
6,-0.330758,-0.152624,-0.219511,-0.214577,-0.695874,-6.978189e-15,-0.916668,-1.664101,1.483993,2.014706,...,-0.223607,1.561911,-0.008395,1.080123e+00,-0.866025,1.809922,1.182640,-0.062257,1.591109,0.979138


In [132]:
df_lv_2007_2013_scaled.to_csv('Data/y/lv_2007_2013_tcdf.csv')

In [133]:
df_lv.columns.equals(df_lv.columns)

True

In [134]:
lv_context_2014_2020 = pd.read_csv("Data/Context/Latvia_2014_2020.csv")

lv_context_2014_2020 = lv_context_2014_2020[["TIME_PERIOD", "EC", "DI", "SSB", "COC", "CPI"]]

df_lv_2014_2020 = df_lv.loc[14:20]

lv_eu_payments_2014_2020 = lv_eu_payments.loc[7:13]

df_lv_2014_2020 = pd.merge(df_lv_2014_2020, lv_context_2014_2020, on='TIME_PERIOD', how='outer')

df_lv_2014_2020 = pd.merge(df_lv_2014_2020, lv_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_lv_2014_2020['EUPC'] = df_lv_2014_2020['EU_Payment_annual'] / df_lv_2014_2020['PS']

df_lv_2014_2020 = df_lv_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [135]:
df_lv_2014_2020

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,LFG,EPFRG,GGAG,PSG,EC,DI,SSB,COC,CPI,EUPC
0,10.85,9.83,19.59,3.45,3.090000,5.350000,7.50,32.0,19.24,14.85,...,-0.01,187.31,-55.0,-0.02,75.590559,7.48,92.332633,0.41,55.0,3.581094e+08
1,9.87,8.64,16.32,3.09,3.310000,5.370000,7.94,32.0,19.52,15.12,...,0.00,107.29,44.0,-0.01,73.237164,7.37,92.038243,0.43,55.0,3.044853e+08
2,9.64,8.40,17.27,1.24,3.200000,5.100000,7.69,31.0,19.78,15.41,...,-0.01,35.05,-165.0,-0.02,71.655374,7.31,94.107515,0.40,57.0,7.912969e+07
3,8.72,7.67,16.97,4.05,3.000000,5.200000,6.87,31.0,20.01,15.67,...,0.00,129.91,-62.0,-0.02,71.655374,7.25,94.112781,0.50,58.0,9.775100e+07
4,7.41,6.42,12.19,1.32,3.500000,5.500000,6.97,31.0,20.20,15.83,...,-0.01,-14.10,536.0,-0.01,73.507226,7.38,94.113949,0.30,58.0,2.796606e+08
5,6.31,5.42,12.41,3.37,3.600000,5.700000,7.29,31.0,20.41,15.94,...,-0.01,17.51,-152.0,-0.02,73.507226,7.49,94.116764,0.48,56.0,3.811824e+08
6,8.10,7.07,14.87,2.81,3.170769,5.380769,7.22,31.0,20.67,16.00,...,0.00,-38.58,-649.0,-0.01,73.507226,7.24,94.137989,0.69,57.0,3.115265e+08


In [136]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_lv_2014_2020.values)

df_lv_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_lv_2014_2020.index, columns=df_lv_2014_2020.columns)

In [137]:
df_lv_2014_2020_scaled.to_csv('Data/y/lv_2014_2020_tcdf.csv')

===================================================================================================================

In [138]:
df_ee = pd.read_csv("Data/y/Estonia.csv")

df_ee = df_ee.loc[6:20]

df_ee

,Unnamed: 0,REF_AREA_LABEL,TIME_PERIOD,HI,LF,UR,URFF,YUA,FDI,QOR,...,EIA,EBSG,GDPC,GDP,EIS,NINA,LBGD,EMP,LTU,FEM
6,6,Estonia,2006,NaN,0.69,5.91,5.59,12.12,10.33,3.68,...,5.063969,-10.108028,10076.523786,2.296909e+10,61.394965,3.219881,3.066797,:,:,NaN
7,7,Estonia,2007,NaN,0.69,4.59,3.76,10.10,13.51,3.78,...,4.776478,-8.790011,12234.211743,2.470806e+10,60.095527,3.837102,3.169094,:,:,:
8,8,Estonia,2008,NaN,0.70,5.45,5.14,12.03,8.12,4.00,...,3.901565,-3.904339,12429.172307,2.344169e+10,60.831163,4.286809,-2.025402,:,:,:
9,9,Estonia,2009,NaN,0.69,13.55,10.30,27.41,9.51,4.22,...,4.054426,4.963764,10590.021094,2.001212e+10,64.503623,4.371720,-2.239512,70.5,3.7,69.9
10,10,Estonia,2010,NaN,0.68,16.71,14.09,32.94,13.28,4.53,...,4.216992,6.420120,11071.755760,2.050237e+10,65.465353,2.214769,-0.609832,67.6,7.5,67.4
11,11,Estonia,2011,NaN,0.69,12.33,11.58,22.39,4.80,4.49,...,4.412399,5.793163,12612.147903,2.206198e+10,63.153093,2.237984,0.526226,71.0,7.0,68.6
12,12,Estonia,2012,NaN,0.68,10.02,9.10,20.86,7.69,4.22,...,4.493347,1.640113,13674.010506,2.287210e+10,64.403281,4.377116,-0.123296,73.1,5.4,71.1
13,13,Estonia,2013,5.43,0.68,8.63,8.15,18.70,4.31,4.20,...,4.262017,2.306589,14539.662078,2.327385e+10,65.464427,3.191025,0.273440,74.1,3.8,71.6
14,14,Estonia,2014,NaN,0.68,7.35,6.81,14.98,6.59,4.39,...,3.858264,3.225526,15492.493600,2.404719e+10,65.993575,3.643242,0.975175,75.0,3.3,71.9
15,15,Estonia,2015,5.43,0.68,6.41,6.32,14.34,-3.07,4.53,...,4.031562,3.561763,15972.915607,2.448921e+10,64.701793,3.682012,0.119938,76.7,2.4,73.7


In [139]:
df_ee = df_ee.copy()
df_ee['LFG'] = df_ee['LF'].diff()
df_ee['EPFRG'] = df_ee['EPFR'].diff()
df_ee['GGAG'] = df_ee['GGA'].diff()
df_ee['PSG'] = df_ee['PS'].diff()
df_ee = df_ee.drop(6)

In [140]:
df_ee = df_ee.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [141]:
df_ee["EMP"] = df_ee["EMP"].replace(':', np.nan)
df_ee["LTU"] = df_ee["LTU"].replace(':', np.nan)
df_ee["FEM"] = df_ee["FEM"].replace(':', np.nan)

In [142]:
missing_values(df_ee)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [143]:
df_ee = df_ee.copy()
fill_with_mode(df_ee, ["QOR", "QOA"])

In [144]:
fill_with_mean(df_ee, ["EMP", "LTU", "FEM"])

In [145]:
ee_context_2007_2013 = pd.read_csv("Data/Context/Estonia_2007_2013.csv")

ee_context_2007_2013 = ee_context_2007_2013[["TIME_PERIOD", "PMSI", "JI", "DBP", "DWCP", "PR"]]

ee_eu_payments = eu_payments[eu_payments['Country'] == 'EE'].copy()

ee_eu_payments['EU_Payment_annual'] = pd.to_numeric(ee_eu_payments['EU_Payment_annual'].str.replace(',', ''))

ee_eu_payments = ee_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

ee_eu_payments = ee_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [146]:
ee_eu_payments_2007_2013 = lv_eu_payments.loc[0:6]

In [147]:
df_ee_2007_2013 = df_ee.loc[7:13]

df_ee_2007_2013 = pd.merge(df_ee_2007_2013, ee_context_2007_2013, on='TIME_PERIOD', how='outer')

df_ee_2007_2013 = pd.merge(df_ee_2007_2013, ee_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

df_ee_2007_2013['EUPC'] = df_ee_2007_2013['EU_Payment_annual'] / df_ee_2007_2013['PS']

df_ee_2007_2013 = df_ee_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [148]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_ee_2007_2013.values)

df_ee_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_ee_2007_2013.index, columns=df_ee_2007_2013.columns)

In [149]:
df_ee_2007_2013_scaled.to_csv('Data/y/ee_2007_2013_tcdf.csv')

In [150]:
df_lv.columns.equals(df_ee.columns)

True

In [151]:
ee_context_2014_2020 = pd.read_csv("Data/Context/Estonia_2014_2020.csv")

ee_context_2014_2020 = ee_context_2014_2020[["TIME_PERIOD", "EC", "DI", "SSB", "COC", "CPI"]]

df_ee_2014_2020 = df_ee.loc[14:20]

ee_eu_payments_2014_2020 = ee_eu_payments.loc[7:13]

df_ee_2014_2020 = pd.merge(df_ee_2014_2020, ee_context_2014_2020, on='TIME_PERIOD', how='outer')

df_ee_2014_2020 = pd.merge(df_ee_2014_2020, ee_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_ee_2014_2020['EUPC'] = df_ee_2014_2020['EU_Payment_annual'] / df_ee_2014_2020['PS']

df_ee_2014_2020 = df_ee_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [152]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_ee_2014_2020.values)

df_ee_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_ee_2014_2020.index, columns=df_ee_2014_2020.columns)

In [153]:
df_ee_2014_2020_scaled.to_csv('Data/y/ee_2014_2020_tcdf.csv')

===================================================================================================================

In [154]:
df_fi = pd.read_csv("Data/y/Finland.csv")

df_fi = df_fi.loc[6:20]

df_fi

,Unnamed: 0,REF_AREA_LABEL,TIME_PERIOD,HI,LF,UR,URFF,YUA,FDI,QOR,...,EIA,EBSG,GDPC,GDP,EIS,NINA,LBGD,EMP,LTU,FEM
6,6,Finland,2006,NaN,2.72,8.94,9.14,25.77,2.14,5.45,...,4.670056,4.165775,32824.193528,2.220008e+11,69.637846,1.520488,4.305193,:,:,NaN
7,7,Finland,2007,NaN,2.68,6.85,7.19,16.43,8.57,5.64,...,4.536961,4.794744,35369.427763,2.337953e+11,69.745498,1.650190,5.292982,:,:,:
8,8,Finland,2008,NaN,2.72,6.37,6.69,16.34,6.79,5.84,...,4.520778,3.615903,36559.083931,2.356292e+11,70.330602,1.635209,4.569814,:,:,:
9,9,Finland,2009,NaN,2.69,8.25,7.57,21.38,-3.48,5.89,...,4.626857,2.038683,34039.968375,2.165997e+11,71.428577,1.816013,-1.860401,72.6,1.7,71.5
10,10,Finland,2010,NaN,2.67,8.39,7.64,21.29,4.90,5.86,...,4.439422,1.364890,35080.114078,2.234622e+11,72.260118,1.599064,-2.369424,71.9,2.5,70.3
11,11,Finland,2011,NaN,2.69,7.78,7.10,19.93,-2.18,5.83,...,4.230618,-0.781159,36682.446617,2.288039e+11,72.774767,1.606380,-0.488730,72.7,2.2,70.7
12,12,Finland,2012,NaN,2.70,7.69,7.05,18.87,1.92,6.09,...,4.159666,-1.560551,37011.280629,2.253195e+11,73.055165,1.762605,-1.104413,73.0,2.1,71.3
13,13,Finland,2013,7.39,2.69,8.19,7.52,19.82,-1.83,6.10,...,4.132272,-0.984781,37414.607025,2.231120e+11,73.020367,1.773043,-1.825088,72.5,2.1,71.0
14,14,Finland,2014,NaN,2.68,8.66,7.95,20.45,6.42,5.87,...,4.255833,-1.015764,37691.943183,2.220454e+11,73.848894,1.721723,-2.267616,72.2,2.4,71.2
15,15,Finland,2015,7.41,2.69,9.38,8.84,22.30,7.54,5.78,...,4.223626,-0.398683,38359.487336,2.230812e+11,74.072208,1.472224,-1.810725,71.8,2.9,70.7


In [155]:
df_fi = df_fi.copy()
df_fi['LFG'] = df_fi['LF'].diff()
df_fi['EPFRG'] = df_fi['EPFR'].diff()
df_fi['GGAG'] = df_fi['GGA'].diff()
df_fi['PSG'] = df_fi['PS'].diff()
df_fi = df_fi.drop(6)

In [156]:
df_fi = df_fi.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [157]:
df_fi["EMP"] = df_fi["EMP"].replace(':', np.nan)
df_fi["LTU"] = df_fi["LTU"].replace(':', np.nan)
df_fi["FEM"] = df_fi["FEM"].replace(':', np.nan)

In [158]:
missing_values(df_fi)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 2 - 14.3 %
LTU - 2 - 14.3 %
FEM - 2 - 14.3 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [159]:
df_fi = df_fi.copy()
fill_with_mode(df_fi, ["QOR", "QOA"])

fill_with_mean(df_fi, ["EMP", "LTU", "FEM"])

In [160]:
fi_context_2007_2013 = pd.read_csv("Data/Context/Finland_2007_2013.csv")

fi_context_2007_2013 = fi_context_2007_2013[["TIME_PERIOD", "DI", "GE", "CPI", "RQ", "VAA"]]

fi_eu_payments = eu_payments[eu_payments['Country'] == 'FI'].copy()

fi_eu_payments['EU_Payment_annual'] = pd.to_numeric(fi_eu_payments['EU_Payment_annual'].str.replace(',', ''))

fi_eu_payments = fi_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

fi_eu_payments = fi_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [161]:
fi_eu_payments_2007_2013 = fi_eu_payments.loc[0:6]

In [162]:
df_fi_2007_2013 = df_fi.loc[7:13]

df_fi_2007_2013 = pd.merge(df_fi_2007_2013, fi_context_2007_2013, on='TIME_PERIOD', how='outer')

df_fi_2007_2013 = pd.merge(df_fi_2007_2013, fi_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

df_fi_2007_2013['EUPC'] = df_fi_2007_2013['EU_Payment_annual'] / df_fi_2007_2013['PS']

df_fi_2007_2013 = df_fi_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [163]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_fi_2007_2013.values)

df_fi_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_fi_2007_2013.index, columns=df_fi_2007_2013.columns)

In [164]:
df_fi_2007_2013_scaled.to_csv('Data/y/fi_2007_2013_tcdf.csv')

In [165]:
df_ee.columns.equals(df_fi.columns)

True

In [166]:
fi_context_2014_2020 = pd.read_csv("Data/Context/Finland_2014_2020.csv")

fi_context_2014_2020 = fi_context_2014_2020[["TIME_PERIOD", "DI", "COC", "CPI", "VAA", "ROL"]]

df_fi_2014_2020 = df_fi.loc[14:20]

fi_eu_payments_2014_2020 = fi_eu_payments.loc[7:13]

df_fi_2014_2020 = pd.merge(df_fi_2014_2020, fi_context_2014_2020, on='TIME_PERIOD', how='outer')

df_fi_2014_2020 = pd.merge(df_fi_2014_2020, fi_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_fi_2014_2020['EUPC'] = df_fi_2014_2020['EU_Payment_annual'] / df_fi_2014_2020['PS']

df_fi_2014_2020 = df_fi_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [167]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_fi_2014_2020.values)

df_fi_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_fi_2014_2020.index, columns=df_fi_2014_2020.columns)

In [168]:
df_fi_2014_2020_scaled.to_csv('Data/y/fi_2014_2020_tcdf.csv')

===================================================================================================================

In [169]:
df_se = pd.read_csv("Data/y/Sweden.csv")

df_se = df_se.loc[6:20]

df_se

,Unnamed: 0,REF_AREA_LABEL,TIME_PERIOD,HI,LF,UR,URFF,YUA,FDI,QOR,...,EIA,EBSG,GDPC,GDP,EIS,NINA,LBGD,EMP,LTU,FEM
6,6,Sweden,2006,NaN,4.77,7.07,7.26,21.48,5.22,5.50,...,2.226741,6.865310,343319.451947,4.093849e+12,75.860085,2.498465,1.972020,78.4,1.0,NaN
7,7,Sweden,2007,NaN,4.84,6.16,6.47,19.26,9.42,5.65,...,2.251167,5.900063,362059.214096,4.225872e+12,76.103068,2.407499,3.214195,79.7,0.8,76.5
8,8,Sweden,2008,NaN,4.90,6.24,6.57,20.21,8.05,5.63,...,2.122367,5.218714,367898.323980,4.186862e+12,76.553322,2.438199,2.032230,80.0,0.8,76.7
9,9,Sweden,2009,NaN,4.91,8.35,8.01,25.02,2.06,5.69,...,2.177660,4.958800,357492.674906,4.008687e+12,77.662551,2.561376,-0.614954,77.9,1.1,75.1
10,10,Sweden,2010,NaN,4.94,8.61,8.52,24.76,0.13,5.71,...,2.100519,4.661454,378703.698372,4.239216e+12,77.996697,2.592290,-0.240938,77.7,1.5,74.4
11,11,Sweden,2011,NaN,5.00,7.80,7.76,22.78,1.22,5.66,...,1.989988,4.436392,392078.684225,4.373341e+12,78.085683,2.433631,0.015331,79.0,1.4,76.0
12,12,Sweden,2012,NaN,5.03,7.98,7.71,23.61,0.78,5.58,...,2.058833,4.367928,391255.979647,4.355215e+12,78.297872,2.455194,-0.914026,79.1,1.4,76.4
13,13,Sweden,2013,7.48,5.09,8.05,7.89,23.52,0.25,5.52,...,2.033299,3.831176,396337.894577,4.404765e+12,78.807199,2.346443,-1.371356,79.4,1.4,76.7
14,14,Sweden,2014,NaN,5.13,7.95,7.70,22.82,-1.05,5.50,...,1.974281,3.093335,408791.051257,4.505888e+12,79.421139,2.229543,-1.129253,79.6,1.3,77.2
15,15,Sweden,2015,7.36,5.18,7.43,7.29,20.36,1.95,5.36,...,2.036107,3.259861,431846.583992,4.704604e+12,79.676015,2.098638,0.377740,80.1,1.3,77.9


In [170]:
df_se = df_se.copy()
df_se['LFG'] = df_se['LF'].diff()
df_se['EPFRG'] = df_se['EPFR'].diff()
df_se['GGAG'] = df_se['GGA'].diff()
df_se['PSG'] = df_se['PS'].diff()
df_se = df_se.drop(6)

In [171]:
df_se = df_se.drop(["Unnamed: 0", "REF_AREA_LABEL", "HI", "OCTB", "NINA", "LF", "EPFR", "GGA"], axis=1)

In [172]:
df_se["EMP"] = df_se["EMP"].replace(':', np.nan)
df_se["LTU"] = df_se["LTU"].replace(':', np.nan)
df_se["FEM"] = df_se["FEM"].replace(':', np.nan)

In [173]:
missing_values(df_se)

TIME_PERIOD - 0 - 0.0 %
UR - 0 - 0.0 %
URFF - 0 - 0.0 %
YUA - 0 - 0.0 %
FDI - 0 - 0.0 %
QOR - 1 - 7.1 %
QOA - 1 - 7.1 %
EIAP - 0 - 0.0 %
PS - 0 - 0.0 %
PDPS - 0 - 0.0 %
P65 - 0 - 0.0 %
P014 - 0 - 0.0 %
PGP - 0 - 0.0 %
VABI - 0 - 0.0 %
GCFP - 0 - 0.0 %
GGFC - 0 - 0.0 %
GDPCG - 0 - 0.0 %
EII - 0 - 0.0 %
EIA - 0 - 0.0 %
EBSG - 0 - 0.0 %
GDPC - 0 - 0.0 %
GDP - 0 - 0.0 %
EIS - 0 - 0.0 %
LBGD - 0 - 0.0 %
EMP - 0 - 0.0 %
LTU - 0 - 0.0 %
FEM - 0 - 0.0 %
LFG - 0 - 0.0 %
EPFRG - 0 - 0.0 %
GGAG - 0 - 0.0 %
PSG - 0 - 0.0 %


In [174]:
df_se = df_se.copy()
fill_with_mode(df_se, ["QOR"])

fill_with_mean(df_se, ["QOA", "FEM"])

In [175]:
se_context_2007_2013 = pd.read_csv("Data/Context/Sweden_2007_2013.csv")

se_context_2007_2013 = se_context_2007_2013[["TIME_PERIOD", "DI", "GE", "CPI", "RQ", "VAA"]]

se_eu_payments = eu_payments[eu_payments['Country'] == 'SE'].copy()

se_eu_payments['EU_Payment_annual'] = pd.to_numeric(se_eu_payments['EU_Payment_annual'].str.replace(',', ''))

se_eu_payments = se_eu_payments.groupby('Year')['EU_Payment_annual'].agg('sum').reset_index()

se_eu_payments = se_eu_payments.rename({"Year":"TIME_PERIOD"}, axis=1)

In [176]:
se_eu_payments_2007_2013 = se_eu_payments.loc[0:6]

In [177]:
df_se_2007_2013 = df_se.loc[7:13]

df_se_2007_2013 = pd.merge(df_se_2007_2013, se_context_2007_2013, on='TIME_PERIOD', how='outer')

df_se_2007_2013 = pd.merge(df_se_2007_2013, se_eu_payments_2007_2013, on='TIME_PERIOD', how='outer')

df_se_2007_2013['EUPC'] = df_se_2007_2013['EU_Payment_annual'] / df_se_2007_2013['PS']

df_se_2007_2013 = df_se_2007_2013.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [178]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_se_2007_2013.values)

df_se_2007_2013_scaled = pd.DataFrame(scaled_features, index=df_se_2007_2013.index, columns=df_se_2007_2013.columns)

In [179]:
df_se_2007_2013_scaled.to_csv('Data/y/se_2007_2013_tcdf.csv')

In [180]:
df_fi.columns.equals(df_se.columns)

True

In [181]:
se_context_2014_2020 = pd.read_csv("Data/Context/Sweden_2014_2020.csv")

se_context_2014_2020 = se_context_2014_2020[["TIME_PERIOD", "DI", "COC", "CPI", "VAA", "ROL"]]

df_se_2014_2020 = df_se.loc[14:20]

se_eu_payments_2014_2020 = se_eu_payments.loc[7:13]

df_se_2014_2020 = pd.merge(df_se_2014_2020, se_context_2014_2020, on='TIME_PERIOD', how='outer')

df_se_2014_2020 = pd.merge(df_se_2014_2020, se_eu_payments_2014_2020, on='TIME_PERIOD', how='outer')

df_se_2014_2020['EUPC'] = df_se_2014_2020['EU_Payment_annual'] / df_se_2014_2020['PS']

df_se_2014_2020 = df_se_2014_2020.drop(['TIME_PERIOD', 'PS', 'EU_Payment_annual'], axis=1)

In [182]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_se_2014_2020.values)

df_se_2014_2020_scaled = pd.DataFrame(scaled_features, index=df_se_2014_2020.index, columns=df_se_2014_2020.columns)

In [183]:
df_se_2014_2020_scaled.to_csv('Data/y/se_2014_2020_tcdf.csv')